### Script generating random samples with a buffer

Script by Shunan Feng, https://github.com/fsn1995/orbit-drift-MODIS-ice-albedo/blob/main/gee/randomSampleBuffer.js, translated into Python and with a few alterations by Simon Kleiner to fit the GEMLST purpose.

In [2]:
import ee
import geemap

geemap.ee_initialize()



In [ ]:
# #
# This is a script to generate random sampling points with buffer to avoid
# the inluence of spatial autocorrelation.

# Shunan Feng (shunan.feng@envs.au.dk)
# #


#
# # random sampling points with strict buffer
#

# region = ee.FeatureCollection("projects/ee-deeppurple/assets/icePoly")
cellSize = 50000
The `seed` value is used in the `randomOffset` function to randomly offset the projection grid and in the `pointsWithBuffer` function to generate random images for sampling points. Changing the `seed` should produce different random outputs each time, as Earth Engine's random functions are deterministic based on the seed.

If the output (e.g., the map layers) appears unchanged, it may be because:
- The cell has not been re-executed after modifying the `seed`. Jupyter Notebook caches cell outputs, so re-run the cell to apply the change.
- The random generation is working, but the visual result looks similar due to the large region or scale; inspect the actual point coordinates or size to confirm differences.

To verify, add a print statement like `print(pointsWithBuffer(grid, region, seed, True).size())` after the map layers to see if the number of points changes with different seeds. If issues persist, check for errors in the Earth Engine console.
projcrs = "EPSG:3413"

greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)

region = greenlandmask.selfMask().reduceToVectors(
  geometryType='polygon',
  reducer=ee.Reducer.countEvery(),
  scale=cellSize,
  maxPixels=1e13
)


# #
# Functions for random sampling with buffer by Noel Gorelick,
# modifed to mask out by ELA.
# ref:https://medium.com/google-earth/random-samples-with-buffering-6c8737384f8c
# #

# Generate random points in the given region, buffered by the scale of the given projection.
# When strict is
#   "True": points will be at least 'scale' apart, with an average spacing of 2*scale.
#   "False": points will be, on average, 'scale' apart, but with no minimum distance guarantee.

def pointsWithBuffer(proj, region, seed, strict):
  # Construct a grid of random numbers with the appropriate sized pixels
  # and randomly offset it, so subsequent runs don't sample from the exact same cells.
  looseGrid = ee.Image.random(seed).multiply(1000000).int()

  # To ensure no points can be closer than the given distance we mask off 8 out of 9 grid cells.
  # leaving only those cells have an odd x and y coordinates.
  # Cell coordinates are centered on the 1/2 pixel. The double not is to avoid float comparison issues.
  mask = ee.Image.pixelCoordinates(proj) \
  .expression('!((b("x") + 0.5) % 2 != 0 or (b("y") + 0.5) % 2 != 0)')
  strictGrid = looseGrid.updateMask(mask)

  # Pick a grid based on the 'strict' option.
  cells = ee.Image(ee.Algorithms.If(strict, strictGrid, looseGrid)).clip(region).reproject(proj)
  # Uncomment to visuaize cells.
  # Map.addLayer(cells.randomVisualizer())

  # Generate another random image and select the maximum random value
  # in each grid cell as the sample point.
  random = ee.Image.random(seed).multiply(1000000).int().clip(region).reproject(proj)
  maximum = cells.addBands(random).reduceConnectedComponents(ee.Reducer.max())

  # Find all the points that are local maximums and convert to a FeatureCollection.
  points = random.eq(maximum).selfMask()
  samples = points.reduceToVectors(
    reducer = ee.Reducer.countEvery(),
    geometry = region,
    crs = proj.scale(1/16, 1/16),
    geometryType = 'centroid',
    maxPixels = 1e13,
  )

  return samples


# Translates a projection by a random amount between 0 and 1 in projection units.

def randomOffset(projection, seed):
  values = ee.FeatureCollection([ee.Feature(None, None)]) \
  .randomColumn('x', seed) \
  .randomColumn('y', seed) \
  .first()
  return projection.translate(values.get("x"), values.get("y"))


# Display the pixel grid assocaited with a projection, as box outlines.

def displayGrid(proj, mask):
  # Scale by 2 because we have 2 zero crossings when using round.
  cells = ee.Image.pixelCoordinates(proj.scale(2,2))
  return cells.subtract(cells.round()).zeroCrossing().reduce('sum').selfMask().updateMask(mask)


#
# # random sampling points with strict buffer
#


grid = randomOffset(ee.Projection(projcrs).atScale(cellSize), seed)

Map = geemap.Map()

Map.addLayer(pointsWithBuffer(grid, region, seed, True), {"color": '#b22222'}, 'Strict')
Map.addLayer(displayGrid(grid, greenlandmask).clip(region), {"palette": ['#92222244']}, 'Strict Grid')
# print(pointsWithBuffer(grid, region, seed, True).size(), f"strict points, spaced {grid.nominalScale().getInfo()} meters apart.")

Map


# Export an ee.FeatureCollection as an Earth Engine asset in case the sampling points size is too big.
# geemap.ee_export_vector_to_asset(
#     "collection"=pointsWithBuffer(grid, region, seed, True),
#     "description" ='randomGrIS5km',
#     "assetId"='projects/ee-deeppurple/assets/orbitdrift/randomGrIS5km',
#   })

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…